In [3]:
!pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 2.2 MB/s eta 0:00:0000:0100:010m


In [12]:
pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.1/165.1 kB 1.3 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [14]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

# Setup
fake = Faker()
random.seed(42)
Faker.seed(42)

# Constants
num_rides = 5000
drivers = [fake.name() for _ in range(50)]
areas = ['KLCC', 'Bangsar', 'Subang Jaya', 'Petaling Jaya', 'Mont Kiara', 'Bukit Bintang']
start_date = datetime(2024, 3, 1)

# Generate Rides Data
rides = []
for i in range(num_rides):
    date = start_date + timedelta(days=random.randint(0, 29))
    hour = random.randint(0, 23)
    pickup_area = random.choice(areas)
    dropoff_area = random.choice([a for a in areas if a != pickup_area])
    wait_time = round(random.uniform(2, 10), 2)
    fare = round(random.uniform(5, 50), 2)
    ride_duration = round(random.uniform(10, 60), 2)
    rating = round(random.uniform(3.5, 5.0), 1)
    driver = random.choice(drivers)
    rides.append([
        i + 1, date.date(), hour, pickup_area, dropoff_area,
        wait_time, fare, ride_duration, rating, driver
    ])

rides_df = pd.DataFrame(rides, columns=[
    "ride_id", "date", "hour", "pickup_area", "dropoff_area", 
    "wait_time", "fare", "ride_duration", "rating", "driver_name"
])

# Drivers Table
drivers_df = pd.DataFrame(list(set(rides_df['driver_name'])), columns=["driver_name"])
drivers_df["driver_id"] = range(1, len(drivers_df) + 1)

# Join driver_id to rides
rides_df = rides_df.merge(drivers_df, on="driver_name")
rides_df = rides_df[[
    "ride_id", "date", "hour", "pickup_area", "dropoff_area",
    "wait_time", "fare", "ride_duration", "rating", "driver_id", "driver_name"
]]

# Daily Summary Table
daily_summary = rides_df.groupby("date").agg(
    total_rides=("ride_id", "count"),
    total_revenue=("fare", "sum"),
    avg_wait_time=("wait_time", "mean")
).reset_index()

# Area Summary Table
area_summary = rides_df.groupby("pickup_area").agg(
    total_rides=("ride_id", "count"),
    avg_fare=("fare", "mean"),
    avg_rating=("rating", "mean")
).reset_index()

# Driver Performance Table
driver_performance = rides_df.groupby(["driver_id", "driver_name"]).agg(
    total_rides=("ride_id", "count"),
    avg_ride_duration=("ride_duration", "mean"),
    avg_rating=("rating", "mean"),
    total_earnings=("fare", "sum")
).reset_index()

# Hourly Distribution Table
hourly_distribution = rides_df.groupby("hour").agg(
    total_rides=("ride_id", "count"),
    avg_wait_time=("wait_time", "mean")
).reset_index()

# Save to Excel
with pd.ExcelWriter("Biba_Dataset.xlsx", engine='xlsxwriter') as writer:
    rides_df.to_excel(writer, index=False, sheet_name="Rides")
    drivers_df.to_excel(writer, index=False, sheet_name="Drivers")
    daily_summary.to_excel(writer, index=False, sheet_name="Daily Summary")
    area_summary.to_excel(writer, index=False, sheet_name="Area Summary")
    driver_performance.to_excel(writer, index=False, sheet_name="Driver Performance")
    hourly_distribution.to_excel(writer, index=False, sheet_name="Hourly Distribution")

print("✅ Biba_Dataset.xlsx created successfully.")

✅ Biba_Dataset.xlsx created successfully.
